In [ ]:
"""
Asymmetric Correlation nach Ang & Chen (2002)
Schematisches Streudiagramm + Exceedance-Korrelationsplot
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

# ── Reproduzierbarkeit ──
np.random.seed(42)

# ── Style (konsistent mit Projekt) ──
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'DejaVu Sans'

# ── Synthetische Daten mit asymmetrischer Abhängigkeit ──
n = 800

# Normalzustand: moderate Korrelation
n_normal = int(n * 0.65)
mean_normal = [0.005, 0.004]
cov_normal = [[0.0004, 0.00012],
              [0.00012, 0.00035]]
r_normal = np.random.multivariate_normal(mean_normal, cov_normal, n_normal)

# Stressregime: hohe Korrelation, negative Renditen
n_stress = n - n_normal
mean_stress = [-0.012, -0.010]
cov_stress = [[0.0012, 0.00090],
              [0.00090, 0.0010]]
r_stress = np.random.multivariate_normal(mean_stress, cov_stress, n_stress)

r_all = np.vstack([r_normal, r_stress])
r_x, r_y = r_all[:, 0], r_all[:, 1]

# ── Exceedance Correlations berechnen ──
thresholds = np.linspace(-0.06, 0.06, 80)
exc_corr_down = []  # corr(x,y | x<t, y<t)
exc_corr_up = []    # corr(x,y | x>t, y>t)

for t in thresholds:
    mask_down = (r_x < t) & (r_y < t)
    mask_up = (r_x > t) & (r_y > t)

    if mask_down.sum() > 10:
        exc_corr_down.append(np.corrcoef(r_x[mask_down], r_y[mask_down])[0, 1])
    else:
        exc_corr_down.append(np.nan)

    if mask_up.sum() > 10:
        exc_corr_up.append(np.corrcoef(r_x[mask_up], r_y[mask_up])[0, 1])
    else:
        exc_corr_up.append(np.nan)

exc_corr_down = np.array(exc_corr_down)
exc_corr_up = np.array(exc_corr_up)

# ── Abbildung ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# ────────────────────────────────────
# Panel (a): Streudiagramm
# ────────────────────────────────────
threshold = 0.0

mask_bear = (r_x < threshold) & (r_y < threshold)
mask_bull = (r_x > threshold) & (r_y > threshold)
mask_mixed = ~mask_bear & ~mask_bull

ax1.scatter(r_x[mask_mixed], r_y[mask_mixed],
            alpha=0.25, s=12, color='gray', label='Gemischt', zorder=2)
ax1.scatter(r_x[mask_bull], r_y[mask_bull],
            alpha=0.4, s=14, color='#2ca02c', label='Aufwärtsmarkt', zorder=3)
ax1.scatter(r_x[mask_bear], r_y[mask_bear],
            alpha=0.5, s=14, color='#d62728', label='Abwärtsmarkt', zorder=3)

# Regressionslinien je Regime
for mask, color, lbl in [
    (mask_bear, '#d62728', 'Abwärts'),
    (mask_bull, '#2ca02c', 'Aufwärts'),
]:
    if mask.sum() > 2:
        slope, intercept, _, _, _ = stats.linregress(r_x[mask], r_y[mask])
        x_fit = np.linspace(r_x[mask].min(), r_x[mask].max(), 100)
        ax1.plot(x_fit, slope * x_fit + intercept,
                 color=color, linewidth=2.0, linestyle='-', zorder=4)

# Korrelationswerte annotieren
rho_bear = np.corrcoef(r_x[mask_bear], r_y[mask_bear])[0, 1]
rho_bull = np.corrcoef(r_x[mask_bull], r_y[mask_bull])[0, 1]

ax1.annotate(f'$\\rho_{{bear}}$ = {rho_bear:.2f}',
             xy=(-0.045, -0.035), fontsize=11, color='#d62728', fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                       edgecolor='#d62728', alpha=0.9))
ax1.annotate(f'$\\rho_{{bull}}$ = {rho_bull:.2f}',
             xy=(0.015, 0.025), fontsize=11, color='#2ca02c', fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                       edgecolor='#2ca02c', alpha=0.9))

ax1.axhline(0, color='black', linewidth=0.8, alpha=0.3)
ax1.axvline(0, color='black', linewidth=0.8, alpha=0.3)
ax1.set_xlabel('Rendite Portfolio X', fontsize=11)
ax1.set_ylabel('Rendite Portfolio Y', fontsize=11)
ax1.set_title('(a) Asymmetrische Korrelationsstruktur', fontsize=12, fontweight='bold', pad=10)
ax1.legend(loc='upper left', fontsize=9, frameon=True)
ax1.grid(True, alpha=0.2)

# ────────────────────────────────────
# Panel (b): Exceedance Correlations
# ────────────────────────────────────
ax2.plot(thresholds, exc_corr_down, color='#d62728', linewidth=2.0,
         label='Abwärts: $\\rho^{exc}(x,y \\mid r < \\xi)$', zorder=3)
ax2.plot(thresholds, exc_corr_up, color='#2ca02c', linewidth=2.0,
         label='Aufwärts: $\\rho^{exc}(x,y \\mid r > \\xi)$', zorder=3)

# Referenzlinie: unconditional correlation
rho_unc = np.corrcoef(r_x, r_y)[0, 1]
ax2.axhline(rho_unc, color='gray', linewidth=1.5, linestyle='--', alpha=0.6,
            label=f'Unbedingte Korrelation ($\\rho$ = {rho_unc:.2f})')

# Asymmetrie-Bereich hervorheben
valid = ~np.isnan(exc_corr_down) & ~np.isnan(exc_corr_up)
mask_neg_t = (thresholds < 0) & valid
ax2.fill_between(thresholds[mask_neg_t],
                 exc_corr_up[mask_neg_t], exc_corr_down[mask_neg_t],
                 alpha=0.15, color='#d62728', label='Asymmetrie-Lücke')

ax2.axvline(0, color='black', linewidth=0.8, alpha=0.3)
ax2.set_xlabel('Schwellenwert $\\xi$', fontsize=11)
ax2.set_ylabel('Exceedance-Korrelation $\\rho^{exc}$', fontsize=11)
ax2.set_title('(b) Exceedance-Korrelation nach Ang & Chen (2002)',
              fontsize=12, fontweight='bold', pad=10)
ax2.legend(loc='lower left', fontsize=9, frameon=True)
ax2.grid(True, alpha=0.2)
ax2.set_ylim(0.0, 1.0)

# ── Layout & Speichern ──
plt.tight_layout()
plt.savefig('../../assets/asymmetric_correlation_ang_chen.png', dpi=300, bbox_inches='tight',
    facecolor='white')
plt.show()

print(f"\nKorrelations-Statistiken:")
print(f"  ρ (unconditional):  {rho_unc:.4f}")
print(f"  ρ (bear, r<0):      {rho_bear:.4f}")
print(f"  ρ (bull, r>0):      {rho_bull:.4f}")
print(f"  Asymmetrie-Ratio:   {rho_bear/rho_bull:.2f}x")